# Reranking and Late Interaction: Cross-Encoders to ColBERT

The full code of [Reranking and Late Interaction: Cross-Encoders to ColBERT](https://residual-stream.lvtan71.workers.dev/posts/reranking-and-late-interaction/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/reranking-and-late-interaction) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/reranking-and-late-interaction.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 35 min end to end.
The post's run read a copy of the blog taken mid-session, with a draft of post 36 that isn't in git; the rebuild post 37 uses has 308 English sections where the post had 309, so the last digits can differ.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" sentence-transformers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "reranking-and-late-interaction"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

In [ ]:
!tar xzf blog-post37.tar.gz

## `cost.py`

Cost per query of a bi-encoder, a cross-encoder over the top 50, and ColBERT's late interaction, for the blog's English sections; the same at 10,000 and 1,000,000 documents; and each design's index size.

In [ ]:
!{sys.executable} cost.py

The post printed:

```text
309 sections, 426 tokens each on average (361 read at most 512); queries average 9.0 tokens
bi-encoder: encode the query, then 309 dot products                198.3 MFLOPs per query
cross-encoder, top 50: 50 passes over query + section          407149.5 MFLOPs per query
late interaction (ColBERT, 128-d): every query x section token      501.3 MFLOPs per query
  roughly K (q + L) / q = 2,056: the cross-encoder's multiple of one query encoding
  at 10,000 documents: bi-encoder   0.21 GFLOPs, MaxSim over every document     10.0 GFLOPs
  at 1,000,000 documents: bi-encoder   0.97 GFLOPs, MaxSim over every document    981.6 GFLOPs

index size, every token indexed, 16-bit numbers:
  bi-encoder, one 384-d vector per section:                 0.23 MiB
  ColBERT, one 128-d vector per token:                     32.12 MiB
  ColBERTv2, 4-byte centroid id + 2-bit residuals:          4.52 MiB
  ColBERTv2, 4-byte centroid id + 1-bit residuals:          2.51 MiB
```

## `maxsim.py`

ColBERT's MaxSim worked on a three-word query against two short documents, with all-MiniLM-L6-v2's token vectors, against a mean of similarities.

In [ ]:
!{sys.executable} maxsim.py

The post printed:

```text
document: "vllm stores the kv cache in fixed-size pages, like virtual memory"
  page    best match 'pages'      0.75   (mean over the document 0.29)
  ##d     best match 'like'       0.46   (mean over the document 0.30)
  kv      best match 'kv'         0.92   (mean over the document 0.30)
  cache   best match 'cache'      0.92   (mean over the document 0.35)
  MaxSim score 3.06; mean-of-similarities score 1.24
document: "the cache line of a cpu holds sixty-four bytes"
  page    best match 'cache'      0.35   (mean over the document 0.22)
  ##d     best match 'cache'      0.36   (mean over the document 0.24)
  kv      best match 'cpu'        0.24   (mean over the document 0.14)
  cache   best match 'cache'      0.87   (mean over the document 0.35)
  MaxSim score 1.81; mean-of-similarities score 0.95
```

## `rerank.py`

Two-stage retrieval on the blog's sections: the best first stage from Semantic and Hybrid Search, then two cross-encoders over its top 50, in English and Vietnamese, with paired MRR differences and the time per query.

In [ ]:
!{sys.executable} rerank.py

The post printed:

```text
English (297 queries), cross-encoder ms-marco-MiniLM-L6-v2:
  stage 1 alone:    recall@1  40%  MRR 0.530  (right section in its top 50: 90%)
  + re-rank top 50: recall@1  45%  MRR 0.575  difference +0.045 [+0.012, +0.079]
  re-ranking time: 1468 ms per query on the CPU (50 pairs)
English (297 queries), cross-encoder ms-marco-MiniLM-L6-v2, best 80-word passage:
  stage 1 alone:    recall@1  40%  MRR 0.530  (right section in its top 50: 90%)
  + re-rank top 50: recall@1  40%  MRR 0.535  difference +0.004 [-0.032, +0.040]
  re-ranking time: 1737 ms per query on the CPU (50 pairs)
English (297 queries), cross-encoder mmarco-mMiniLMv2-L12-H384-v1:
  stage 1 alone:    recall@1  40%  MRR 0.530  (right section in its top 50: 90%)
  + re-rank top 50: recall@1  39%  MRR 0.516  difference -0.014 [-0.054, +0.025]
  re-ranking time: 2377 ms per query on the CPU (50 pairs)
Vietnamese (257 queries), cross-encoder mmarco-mMiniLMv2-L12-H384-v1:
  stage 1 alone:    recall@1  42%  MRR 0.539  (right section in its top 50: 91%)
  + re-rank top 50: recall@1  41%  MRR 0.529  difference -0.010 [-0.055, +0.037]
  re-ranking time: 2317 ms per query on the CPU (50 pairs)
Vietnamese (257 queries), cross-encoder ms-marco-MiniLM-L6-v2:
  stage 1 alone:    recall@1  42%  MRR 0.539  (right section in its top 50: 91%)
  + re-rank top 50: recall@1  35%  MRR 0.487  difference -0.053 [-0.098, -0.007]
  re-ranking time: 976 ms per query on the CPU (50 pairs)
```